# Update Ground Truth with Predicted Document Types

Reads batch results CSV and writes the model's detected `DOCUMENT_TYPE` into the ground truth CSV.

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
# --- Configure paths ---
GROUND_TRUTH_CSV = Path("../evaluation_data/bank/ground_truth_bank.csv")
RESULTS_CSV = Path("../evaluation_data/output/batch")  # directory containing batch_results_*.csv

# Auto-find latest results CSV
results_files = sorted(RESULTS_CSV.glob("batch_results_*.csv"))
assert results_files, f"No batch_results_*.csv found in {RESULTS_CSV}"
RESULTS_CSV = results_files[-1]
print(f"Ground truth: {GROUND_TRUTH_CSV}")
print(f"Results CSV:  {RESULTS_CSV}")

In [ ]:
# Load both CSVs
gt = pd.read_csv(GROUND_TRUTH_CSV, dtype=str)
results = pd.read_csv(RESULTS_CSV, dtype=str)

# Find image column in ground truth
image_col = next(c for c in ["image_file", "filename", "image_name", "file"] if c in gt.columns)
print(f"Image column: {image_col}")
print(f"Ground truth: {len(gt)} rows")
print(f"Results: {len(results)} rows")

In [ ]:
# Map predicted document types to ground truth
type_map = results.set_index("image_name")["document_type"]
gt["DOCUMENT_TYPE"] = gt[image_col].map(type_map)

# Show mapping results
print("Document type distribution:")
print(gt["DOCUMENT_TYPE"].value_counts())
print(f"\nUnmapped: {gt['DOCUMENT_TYPE'].isna().sum()}")

In [ ]:
# Preview
gt[[image_col, "DOCUMENT_TYPE"]].head(10)

In [ ]:
# Save updated ground truth
output_path = GROUND_TRUTH_CSV.parent / f"{GROUND_TRUTH_CSV.stem}_with_types.csv"
gt.to_csv(output_path, index=False)
print(f"Saved: {output_path}")